# Seismic Vulnerability Assessment — LLM-Orchestrated Workflow
### UTS Engineering Graduate Project PG (42003)
**Student:** Kabish Jung Thapa (25631413) | **Supervisor:** Prof. Jianchun Li

**Run order:** Execute cells top-to-bottom (Shift+Enter each).  
Cell 1 installs packages (~60 s). Cells 2–5 define everything. Cell 6 runs all three buildings.

**No API key required.** Keyword-based demo extraction. OpenSeesPy nonlinear analysis.

---
| Cell | Purpose |
|------|---------|
| 1 | Install + imports |
| 2 | Constants, era defaults, case study definitions |
| 3 | Parameter extraction + human verification |
| 4 | OpenSeesPy model: static, eigenvalue, pushover, time-history |
| 5 | Post-processing: EDPs, damage states, fragility curves |
| 6 | **RUN ALL THREE BUILDINGS** — plots + report |


## Cell 1 — Install packages and import libraries
Run this first. Takes ~60 seconds on a fresh Colab runtime.

In [ ]:
# Cell 1: Install and import
import subprocess, sys
subprocess.run(['pip', 'install', 'openseespy', 'numpy', 'matplotlib', 'scipy', '-q'], check=True)

# Force fresh import in case of stale Colab state
for mod in list(sys.modules.keys()):
    if 'opensees' in mod:
        del sys.modules[mod]

import openseespy.opensees as ops
import numpy as np
import scipy.stats as stats
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from matplotlib.patches import Patch, FancyArrowPatch
from matplotlib.lines import Line2D
import tempfile, os, json, re, time, warnings
from datetime import datetime

warnings.filterwarnings('ignore')
print("✓ openseespy  ✓ numpy  ✓ matplotlib  ✓ scipy")
print("All packages ready. Proceed to Cell 2.")

## Cell 2 — Constants, era defaults, and case study definitions
Defines all structural parameters for the three verified case study buildings.  
Edit values here to test different scenarios before running Cell 6.

In [ ]:
# Cell 2: Constants, era defaults, and three verified case study buildings

# ── Physical constants ────────────────────────────────────────────────────────
G            = 9.81     # m/s²
COVER        = 0.040    # m  concrete cover to steel centreline
DRIFT_LIMIT  = 0.015    # 1.5% AS1170.4 Cl 5.4.4

# ── HAZUS-MH damage state thresholds (FEMA 2003, adapted for RC frames) ──────
# Each tuple: (label, lower_PIDR, upper_PIDR, repair_cost_ratio)
DAMAGE_STATES = [
    ("None",      0.000, 0.005, 0.00),
    ("Slight",    0.005, 0.010, 0.05),
    ("Moderate",  0.010, 0.020, 0.20),   # AS1170.4 limit falls here
    ("Extensive", 0.020, 0.040, 0.50),
    ("Complete",  0.040, 9.999, 1.00),
]

# Lognormal fragility parameters (median PIDR, log-std) per damage state
# Source: Adapted from HAZUS-MH Table 5.9 for low-rise non-ductile RC
FRAGILITY = {
    "pre-1990":  {"Slight":   (0.005, 0.60),
                  "Moderate": (0.010, 0.60),
                  "Extensive":(0.020, 0.60),
                  "Complete": (0.040, 0.60)},
    "post-1990": {"Slight":   (0.007, 0.55),
                  "Moderate": (0.014, 0.55),
                  "Extensive":(0.028, 0.55),
                  "Complete": (0.056, 0.55)},
    "post-2010": {"Slight":   (0.010, 0.50),
                  "Moderate": (0.020, 0.50),
                  "Extensive":(0.040, 0.50),
                  "Complete": (0.080, 0.50)},
}

ERA_COLOURS  = {'pre-1990':'#d9534f', 'post-1990':'#f0ad4e', 'post-2010':'#5cb85c'}
STATE_COLOURS= {'None':'#2ecc71','Slight':'#f1c40f',
                'Moderate':'#e67e22','Extensive':'#e74c3c','Complete':'#8e44ad'}

ERA_DEFAULTS = {
    "pre-1990":  dict(fc=20.0,  fy=250.0, col_b=0.30, col_h=0.30,
                      beam_b=0.30, beam_h=0.45, col_rho=0.015,
                      beam_rho_t=0.008,  beam_rho_c=0.004,
                      mu=2.0, Sp=0.77, epsc0=-0.004, epsU=-0.012),
    "post-1990": dict(fc=32.0,  fy=500.0, col_b=0.35, col_h=0.35,
                      beam_b=0.30, beam_h=0.50, col_rho=0.020,
                      beam_rho_t=0.012,  beam_rho_c=0.006,
                      mu=3.0, Sp=0.67, epsc0=-0.005, epsU=-0.020),
    "post-2010": dict(fc=40.0,  fy=500.0, col_b=0.40, col_h=0.40,
                      beam_b=0.35, beam_h=0.55, col_rho=0.025,
                      beam_rho_t=0.015,  beam_rho_c=0.0075,
                      mu=4.0, Sp=0.67, epsc0=-0.006, epsU=-0.030),
}

HAZARD_FACTORS = dict(
    newcastle=0.11, sydney=0.08, wollongong=0.08, nsw=0.11,
    melbourne=0.08, brisbane=0.05, adelaide=0.10,
    perth=0.09, canberra=0.08, hobart=0.05, darwin=0.09,
)

PARAM_BOUNDS = dict(
    num_storeys=(1,4), storey_height=(2.4,4.5), num_bays=(1,6),
    bay_width=(2.5,8.0), fc=(15.0,65.0), fy=(200.0,600.0),
    col_b=(0.20,0.80), col_h=(0.20,0.80), beam_b=(0.20,0.60),
    beam_h=(0.25,0.80), col_rho=(0.01,0.04), mu=(1.5,6.0), Z=(0.03,0.45),
)

# ── Three verified case study buildings ───────────────────────────────────────
# All buildings: 2 storeys, 3 bays x 4.0 m, floor width 8 m
# Location: Newcastle, NSW  (Z=0.11, Site De)
# Dead load: 5.0 kPa  |  Live load: 2.0 kPa

BUILDINGS = [
    dict(
        building_name  = "Building 1 -- Pre-1990 Non-Ductile RC Frame",
        era            = "pre-1990",
        num_storeys=2, storey_height=3.0, num_bays=3, bay_width=4.0,
        floor_width=8.0, fc=20.0, fy=250.0,
        col_b=0.30, col_h=0.30, beam_b=0.30, beam_h=0.45,
        col_rho=0.015, beam_rho_t=0.008, beam_rho_c=0.004,
        mu=2.0, Sp=0.77, Z=0.11, site_class="De",
        dead_load=5.0, live_load=2.0, confidence=1.0,
        epsc0_core=-0.004, epsU_core=-0.012,
        assumptions=["Verified case study -- parameters from thesis Table 2"],
    ),
    dict(
        building_name  = "Building 2 -- Post-1990 Ductile RC Frame",
        era            = "post-1990",
        num_storeys=2, storey_height=3.0, num_bays=3, bay_width=4.0,
        floor_width=8.0, fc=32.0, fy=500.0,
        col_b=0.35, col_h=0.35, beam_b=0.30, beam_h=0.50,
        col_rho=0.020, beam_rho_t=0.012, beam_rho_c=0.006,
        mu=3.0, Sp=0.67, Z=0.11, site_class="De",
        dead_load=5.0, live_load=2.0, confidence=1.0,
        epsc0_core=-0.005, epsU_core=-0.020,
        assumptions=["Verified case study -- parameters from thesis Table 2"],
    ),
    dict(
        building_name  = "Building 3 -- Post-2010 Code-Compliant RC Frame",
        era            = "post-2010",
        num_storeys=2, storey_height=3.0, num_bays=3, bay_width=4.0,
        floor_width=8.0, fc=40.0, fy=500.0,
        col_b=0.40, col_h=0.40, beam_b=0.35, beam_h=0.55,
        col_rho=0.025, beam_rho_t=0.015, beam_rho_c=0.0075,
        mu=4.0, Sp=0.67, Z=0.11, site_class="De",
        dead_load=5.0, live_load=2.0, confidence=1.0,
        epsc0_core=-0.006, epsU_core=-0.030,
        assumptions=["Verified case study -- parameters from thesis Table 2"],
    ),
]

def classify_damage(pidr):
    for name, lo, hi, _ in DAMAGE_STATES:
        if lo <= pidr < hi:
            return name
    return "Complete"

def fragility_probability(era, pidr_value):
    """P(DS >= ds | PIDR=pidr) for each damage state using lognormal CDF."""
    frag = FRAGILITY.get(era, FRAGILITY["pre-1990"])
    result = {}
    for ds, (median, beta) in frag.items():
        if pidr_value > 0 and median > 0:
            z = np.log(pidr_value / median) / beta
            result[ds] = float(stats.norm.cdf(z))
        else:
            result[ds] = 0.0
    return result

print("✓ Constants and building definitions loaded.")
print("  Buildings defined:", len(BUILDINGS))
for b in BUILDINGS:
    print("  |-- {} | f'c={} MPa | mu={}".format(
          b['building_name'][:50], b['fc'], b['mu']))

## Cell 3 — Parameter extraction and human verification
Contains keyword-based extractor (demo mode) and verification checkpoint.  
For the preset buildings in Cell 6, verification is bypassed automatically.  
For custom buildings, you will be prompted to approve/edit parameters.

In [ ]:
# Cell 3: Parameter extraction and human verification checkpoint

# ── Extraction ────────────────────────────────────────────────────────────────

def extract_parameters(description):
    """Keyword-based extraction -- simulates Claude/GPT extraction without API key."""
    d = description.lower()
    assumptions = []

    # Era
    pre_kws  = [str(y) for y in range(1960,1990)] + [
        'pre-1990','old','heritage','brick veneer','unreinforced','non-ductile']
    post10   = [str(y) for y in range(2011,2026)] + [
        'post-2010','modern','new build','recently built','fully ductile']
    post90   = [str(y) for y in range(1990,2011)] + ['post-1990']

    if   any(k in d for k in pre_kws):  era = "pre-1990"
    elif any(k in d for k in post10):   era = "post-2010"
    elif any(k in d for k in post90):   era = "post-1990"
    else:
        era = "pre-1990"
        assumptions.append("Era not stated -- defaulted to pre-1990 (conservative)")

    props = dict(ERA_DEFAULTS[era])
    assumptions.append("Properties defaulted from {} era".format(era))

    # Storeys
    storey_kw = {1:['one storey','1 storey','1-storey','single storey'],
                 2:['two storey','2 storey','2-storey'],
                 3:['three storey','3 storey','3-storey'],
                 4:['four storey','4 storey','4-storey']}
    num_storeys = 2
    for n,kws in storey_kw.items():
        if any(k in d for k in kws): num_storeys=n; break
    m = re.search(r'(\d)\s*(?:floor|level)', d)
    if m: num_storeys=int(m.group(1))
    if 'storey' not in d and 'floor' not in d:
        assumptions.append("Storeys not stated -- defaulted to 2")

    # Geometry
    fl, fw, nb, bw = 12.0, 8.0, 3, 4.0
    for pat in [r'(\d+(?:\.\d+)?)\s*(?:m|metres?)?\s*(?:x|by|x)\s*(\d+(?:\.\d+)?)',
                r'(\d+(?:\.\d+)?)\s*x\s*(\d+(?:\.\d+)?)']:
        m = re.search(pat, d)
        if m:
            l,w = float(m.group(1)),float(m.group(2))
            if w>l: l,w=w,l
            fl,fw = l,w; nb=max(2,round(l/4)); bw=l/nb; break
    else:
        assumptions.append("Floor plan not stated -- defaulted to 12m x 8m")

    # Zone
    Z, sc = 0.11, 'De'
    city_map = {'newcastle':0.11,'sydney':0.08,'melbourne':0.08,
                'brisbane':0.05,'adelaide':0.10,'perth':0.09,'canberra':0.08}
    for city,z in city_map.items():
        if city in d: Z=z; sc='De' if city in ('newcastle','sydney') else 'Ce'; break
    else:
        assumptions.append("Location not identified -- defaulted to Newcastle Z=0.11")

    return dict(
        building_name="{} RC Frame ({}-storey)".format(era.capitalize(), num_storeys),
        num_storeys=num_storeys, storey_height=3.0, num_bays=nb, bay_width=round(bw,2),
        floor_width=fw, fc=props['fc'], fy=props['fy'],
        col_b=props['col_b'], col_h=props['col_h'],
        beam_b=props['beam_b'], beam_h=props['beam_h'],
        col_rho=props['col_rho'], beam_rho_t=props['beam_rho_t'],
        beam_rho_c=props['beam_rho_c'],
        epsc0_core=props['epsc0'], epsU_core=props['epsU'],
        mu=props['mu'], Sp=props['Sp'], Z=Z, site_class=sc,
        dead_load=5.0, live_load=2.0, era=era, confidence=0.75,
        assumptions=assumptions,
    )


# ── Validation ────────────────────────────────────────────────────────────────

def validate_params(params):
    """Return list of warning strings for out-of-range parameters."""
    warnings_list = []
    for key,(lo,hi) in PARAM_BOUNDS.items():
        val = params.get(key)
        if val is None:
            warnings_list.append("MISSING: {}".format(key))
        elif not (lo <= float(val) <= hi):
            warnings_list.append("OUT OF RANGE: {}={} (expected {}-{})".format(key,val,lo,hi))
    if params.get('beam_h',0) <= params.get('beam_b',1):
        warnings_list.append("GEOMETRY: beam depth should exceed width")
    return warnings_list


# ── Verification Checkpoint ───────────────────────────────────────────────────

def verification_checkpoint(params, auto_approve=False):
    """
    Show all extracted parameters and require engineer sign-off.
    This is the human-in-the-loop gate -- no analysis runs without approval.
    auto_approve=True is used only for the preset verified case studies.
    """
    p = params
    print()
    print("=" * 65)
    print("  HUMAN VERIFICATION CHECKPOINT")
    print("=" * 65)
    print("  Building   : {}".format(p['building_name']))
    print("  Era        : {}  |  Confidence: {:.0%}".format(
          p['era'], p.get('confidence',0)))
    print()
    print("  Geometry:")
    print("    {} storeys x {:.1f} m high".format(p['num_storeys'], p['storey_height']))
    print("    {} bays x {:.2f} m wide, frame width {:.1f} m".format(
          p['num_bays'], p['bay_width'], p['floor_width']))
    print()
    print("  Materials:")
    print("    Concrete f'c = {} MPa  |  Steel fy = {} MPa".format(p['fc'],p['fy']))
    print()
    print("  Members:")
    print("    Columns  {}x{} mm  rho={:.1f}%".format(
          int(p['col_b']*1000), int(p['col_h']*1000), p['col_rho']*100))
    print("    Beams    {}x{} mm  rho_t={:.1f}%".format(
          int(p['beam_b']*1000), int(p['beam_h']*1000), p['beam_rho_t']*100))
    print()
    print("  Seismic (AS1170.4:2007):")
    print("    Z={}, Site={}, mu={}, Sp={}".format(
          p['Z'], p['site_class'], p['mu'], p['Sp']))
    print()
    if p.get('assumptions'):
        print("  Assumptions made:")
        for a in p['assumptions']:
            print("    [!] {}".format(a))
        print()

    warnings_list = validate_params(params)
    if warnings_list:
        print("  VALIDATION WARNINGS:")
        for w in warnings_list:
            print("    [W] {}".format(w))
        print()

    if auto_approve:
        print("  [auto-approved -- verified case study]")
        return params, True

    print("  [Y] Approve and run  [E] Edit a value  [N] Reject")
    while True:
        c = input("  Choice: ").strip().upper()
        if c == 'Y':
            print("  Approved.")
            return params, True
        elif c == 'E':
            params = _edit_one_param(params)
            return verification_checkpoint(params, False)
        elif c == 'N':
            print("  Rejected.")
            return params, False
        print("  Enter Y, E or N.")


def _edit_one_param(params):
    print("  Editable keys:", ", ".join(PARAM_BOUNDS.keys()))
    key = input("  Parameter to edit: ").strip()
    if key not in params:
        print("  Not found.")
        return params
    current = params[key]
    raw = input("  {} [{}] => ".format(key, current)).strip()
    try:
        params[key] = int(raw) if isinstance(current,int) else float(raw)
        params.setdefault('assumptions',[]).append(
            "User set {}={}".format(key, params[key]))
        print("  Updated: {}={}".format(key, params[key]))
    except ValueError:
        print("  Invalid -- keeping {}={}".format(key, current))
    return params

print("✓ Extraction and verification functions loaded.")

## Cell 4 — OpenSeesPy analysis engine
Four analysis functions: static base shear, eigenvalue, pushover, time-history.

**Key technical notes documented inline:**
- `fullGenLapack` required with `equalDOF` — ARPACK fails (undocumented bug)
- Masses on master nodes only — slave nodes get zero mass
- `constraints('Transformation')` required for transient with `equalDOF`
- Convergence fallback chain: Newton → KrylovNewton → ModifiedNewton

In [ ]:
# Cell 4: OpenSeesPy analysis engine

import traceback

# ── Model builder ─────────────────────────────────────────────────────────────

def build_model(p):
    """
    Build 2D nonlinear fibre RC frame and run gravity.
    Returns (node_id, M_floor, W_total, convergence_log).

    Node numbering:  (floor+1)*10 + (col+1)
    Ground: 11 12 13 14   Floor 1: 21 22 23 24   Roof: 31 32 33 34
    Master node = leftmost column (node_id[floor][0])
    """
    conv_log = []

    fc_kN  = p['fc']  * 1000
    fy_kN  = p['fy']  * 1000
    Es_kN  = 200000.0 * 1000
    Ac     = p['col_b']  * p['col_h']
    Asc    = p['col_rho']    * Ac
    Ab     = p['beam_b'] * p['beam_h']
    Ast    = p['beam_rho_t'] * Ab
    Asc2   = p['beam_rho_c'] * Ab

    fl_area = p['num_bays'] * p['bay_width'] * p['floor_width']
    W_floor = (p['dead_load'] + 0.3*p['live_load']) * fl_area
    W_total = W_floor * p['num_storeys']
    M_floor = W_floor / G
    P_int   = (p['dead_load']+p['live_load']) * p['floor_width']/2 * p['bay_width']
    P_ext   = P_int / 2

    ops.wipe()
    ops.model('basic', '-ndm', 2, '-ndf', 3)

    xs = [j * p['bay_width']     for j in range(p['num_bays']+1)]
    ys = [i * p['storey_height'] for i in range(p['num_storeys']+1)]

    node_id = []
    for fi,y in enumerate(ys):
        row = []
        for ci,x in enumerate(xs):
            nid = (fi+1)*10 + (ci+1)
            ops.node(nid, x, y)
            row.append(nid)
        node_id.append(row)

    for nid in node_id[0]:
        ops.fix(nid, 1, 1, 1)

    # Rigid diaphragm -- DOF 1 (X) constrained
    for fi in range(1, len(ys)):
        master = node_id[fi][0]
        for slave in node_id[fi][1:]:
            ops.equalDOF(master, slave, 1)

    # Materials
    epsc0 = p.get('epsc0_core', -0.004)
    epsU  = p.get('epsU_core',  -0.012)
    ops.uniaxialMaterial('Concrete01', 1, -fc_kN, epsc0, -0.2*fc_kN, epsU)
    ops.uniaxialMaterial('Concrete01', 2, -fc_kN, -0.002, 0.0, -0.004)
    ops.uniaxialMaterial('Steel01',    3,  fy_kN, Es_kN,  0.01)

    # Column fibre section (tag=1)
    cy = p['col_h']/2 - COVER;  cz = p['col_b']/2 - COVER
    ops.section('Fiber', 1)
    ops.patch('rect', 1, 10, 10, -cy, -cz,  cy,  cz)
    ops.patch('rect', 2, 10,  2,  cy, -p['col_b']/2,  p['col_h']/2, p['col_b']/2)
    ops.patch('rect', 2, 10,  2, -p['col_h']/2, -p['col_b']/2, -cy, p['col_b']/2)
    ops.patch('rect', 2,  2, 10, -cy, -p['col_b']/2, cy, -cz)
    ops.patch('rect', 2,  2, 10, -cy,  cz,            cy, p['col_b']/2)
    As_bar = max(Asc/6, 1e-5)
    ops.layer('straight', 3, 3, As_bar, -cy, -cz, -cy, cz)
    ops.layer('straight', 3, 3, As_bar,  cy, -cz,  cy, cz)

    # Beam fibre section (tag=2)
    by = p['beam_h']/2 - COVER;  bz = p['beam_b']/2 - COVER
    ops.section('Fiber', 2)
    ops.patch('rect', 1, 10, 10, -by, -bz, by, bz)
    ops.patch('rect', 2, 10,  2,  by, -p['beam_b']/2, p['beam_h']/2, p['beam_b']/2)
    ops.patch('rect', 2, 10,  2, -p['beam_h']/2, -p['beam_b']/2, -by, p['beam_b']/2)
    ops.layer('straight', 3, 3, Ast/3,  -by, -bz, -by, bz)
    ops.layer('straight', 3, 3, Asc2/3,  by, -bz,  by, bz)

    ops.geomTransf('PDelta', 1)
    ops.geomTransf('Linear', 2)

    eid = 100
    for fi in range(p['num_storeys']):
        for ci in range(p['num_bays']+1):
            ops.element('nonlinearBeamColumn', eid,
                        node_id[fi][ci], node_id[fi+1][ci], 5, 1, 1)
            eid += 1
    for fi in range(1, p['num_storeys']+1):
        for ci in range(p['num_bays']):
            ops.element('nonlinearBeamColumn', eid,
                        node_id[fi][ci], node_id[fi][ci+1], 5, 2, 2)
            eid += 1

    # Gravity analysis
    ops.timeSeries('Constant', 1);  ops.pattern('Plain', 1, 1)
    for fi in range(1, p['num_storeys']+1):
        for ci,nid in enumerate(node_id[fi]):
            P = -P_ext if (ci==0 or ci==p['num_bays']) else -P_int
            ops.load(nid, 0.0, P, 0.0)
    ops.system('BandGeneral');   ops.numberer('RCM')
    ops.constraints('Plain');    ops.integrator('LoadControl', 0.1)
    ops.algorithm('Newton');     ops.analysis('Static')
    ok = ops.analyze(10)
    conv_log.append({'stage':'gravity', 'converged': ok==0})
    ops.loadConst('-time', 0.0)

    # Mass assignment -- master nodes ONLY
    # CRITICAL: ARPACK fails with mass on slave nodes + equalDOF
    for fi in range(1, p['num_storeys']+1):
        ops.mass(node_id[fi][0], M_floor, M_floor, 0.0)

    return node_id, M_floor, W_total, conv_log


# ── Static analysis ───────────────────────────────────────────────────────────

def static_analysis(p, W_total):
    """AS1170.4:2007 equivalent static base shear -- Cl 6.2"""
    Hn = p['num_storeys'] * p['storey_height']
    T1 = 0.075 * Hn**0.75         # Appendix B approximation

    if   T1 <= 0.10: Ch = 2.35
    elif T1 <  1.50: Ch = 1.65 * (0.1/T1)**0.85
    else:            Ch = 1.10 * (1.5/T1)**2.0

    V = max((p['Z']/p['mu']) * p['Sp'] * Ch * W_total, 0.01*W_total)
    return {'V_static':V, 'Ch':Ch, 'T1_code':T1,
            'Hn':Hn, 'V_over_W':V/W_total}


# ── Eigenvalue analysis ───────────────────────────────────────────────────────

def eigenvalue_analysis(p):
    """
    fullGenLapack solver -- required when equalDOF constraints present.
    ARPACK (default) fails with 'Starting vector is zero' in this configuration.
    See: github.com/YOUR_REPO/blob/main/docs/KNOWN_ISSUES.md
    """
    try:
        eigs   = ops.eigen('-fullGenLapack', p['num_storeys'])
        omega1 = abs(eigs[0])**0.5
        omega2 = abs(eigs[1])**0.5 if p['num_storeys']>=2 else omega1*3
        T1 = 2*np.pi/omega1;  T2 = 2*np.pi/omega2
        return {'T1':T1, 'T2':T2, 'omega1':omega1, 'omega2':omega2,
                'eigs':eigs, 'success':True, 'error':None}
    except Exception as e:
        return {'T1':None,'T2':None,'omega1':None,'omega2':None,
                'eigs':None,'success':False,'error':str(e)}


# ── Pushover analysis ─────────────────────────────────────────────────────────

def run_pushover(node_id, p, W_total, V_static):
    """
    Displacement-controlled pushover to 3% roof drift.
    Load pattern: inverted triangle proportional to storey height.
    Returns capacity curve arrays, V_max, ductility, convergence log.
    """
    conv_log = []
    Hn       = p['num_storeys'] * p['storey_height']
    target   = 0.03 * Hn    # 3% roof drift
    n_steps  = 200

    # Inverted triangle load distribution
    heights    = [(i+1)*p['storey_height'] for i in range(p['num_storeys'])]
    fractions  = [h/sum(heights) for h in heights]

    ops.timeSeries('Linear', 10)
    ops.pattern('Plain', 10, 10)
    for fi,frac in enumerate(fractions, 1):
        ops.load(node_id[fi][0], frac, 0.0, 0.0)

    ops.system('UmfPack');          ops.numberer('RCM')
    ops.constraints('Transformation')
    ops.test('NormDispIncr', 1e-6, 150, 0)
    ops.algorithm('Newton')
    ops.integrator('DisplacementControl', node_id[-1][0], 1, target/n_steps)
    ops.analysis('Static')

    roof_disp  = [0.0];  base_shear = [0.0]
    n_fail_push = 0

    for step in range(n_steps):
        ok = ops.analyze(1)
        if ok != 0:
            n_fail_push += 1
            for alg in [('KrylovNewton',), ('ModifiedNewton','-initial')]:
                ops.algorithm(*alg)
                ok = ops.analyze(1)
                ops.algorithm('Newton')
                if ok == 0: break
        if ok != 0:
            conv_log.append({'step':step,'status':'non_convergence_stop'})
            break

        d = ops.nodeDisp(node_id[-1][0], 1)
        V = sum(-ops.nodeReaction(nid, 1) for nid in node_id[0])
        roof_disp.append(d)
        base_shear.append(V)

    rd = np.array(roof_disp)
    bs = np.array(base_shear)

    # Performance point calculations
    V_max   = float(np.max(bs)) if len(bs)>1 else V_static
    V_60pct = 0.60 * V_max
    d_yield = next((rd[i] for i,v in enumerate(bs) if v>=V_60pct), None)
    ductility = float(rd[-1]/d_yield) if (d_yield and d_yield>0) else None

    # Equivalent linearisation -- effective period
    k_eff   = V_max / rd[-1] if rd[-1]>0 else 0
    T_eff   = 2*np.pi * np.sqrt(W_total/(G * k_eff)) if k_eff>0 else None

    conv_log.append({'stage':'pushover','n_steps_complete':len(rd)-1,
                     'n_fail':n_fail_push,'target_drift_pct':3.0})
    return dict(
        roof_disp   = rd,
        base_shear  = bs,
        roof_drift  = rd/Hn*100,
        V_max       = V_max,
        d_yield_m   = d_yield,
        ductility   = ductility,
        T_eff_s     = T_eff,
        n_fail      = n_fail_push,
        conv_log    = conv_log,
    )


# ── Time-history analysis ─────────────────────────────────────────────────────

def run_time_history(node_id, p, omega1, omega2):
    """
    Nonlinear transient analysis under synthetic sine-wave ground motion.
    Rayleigh damping 5% at modes 1 and 2.
    Sub-step dt=0.005 s  (half the GM time step).
    Fallback chain: Newton -> KrylovNewton -> ModifiedNewton(-initial).
    """
    conv_log = []
    xi = 0.05
    a0 = xi * 2*omega1*omega2 / (omega1+omega2)
    a1 = xi * 2 / (omega1+omega2)
    ops.rayleigh(a0, 0.0, 0.0, a1)

    T1   = 2*np.pi/omega1
    dt   = 0.01;  duration = 20.0
    t    = np.arange(0, duration, dt)
    freq = min(1.0/T1, 4.0)
    env  = np.sin(np.pi*t/duration)
    accel= p['Z']*G*env*np.sin(2*np.pi*freq*t)

    tmp = tempfile.NamedTemporaryFile(mode='w', suffix='.txt', delete=False)
    np.savetxt(tmp.name, accel, fmt='%.8f');  tmp.close()

    ops.timeSeries('Path', 2, '-dt', dt, '-filePath', tmp.name, '-factor', 1.0)
    ops.pattern('UniformExcitation', 2, 1, '-accel', 2)
    ops.system('UmfPack');   ops.numberer('RCM')
    ops.constraints('Transformation')    # REQUIRED with equalDOF
    ops.test('NormDispIncr', 1e-8, 10, 0)
    ops.integrator('Newmark', 0.5, 0.25) # average acceleration
    ops.algorithm('Newton')
    ops.analysis('Transient')

    dt_sub  = dt/2;  n_steps = int(len(t)*dt/dt_sub)
    time_h,dg,df,dr = [],[],[],[]
    n_fail = 0;  n_newton=0;  n_krylov=0;  n_modified=0

    for _ in range(n_steps):
        ok = ops.analyze(1, dt_sub);  n_newton+=1
        if ok != 0:
            n_fail += 1
            ops.algorithm('KrylovNewton');  n_krylov+=1
            ok = ops.analyze(1, dt_sub/5)
            if ok != 0:
                ops.test('NormDispIncr',1e-6,100,0)
                ops.algorithm('ModifiedNewton','-initial');  n_modified+=1
                ops.analyze(1, dt_sub/10)
            ops.algorithm('Newton')
            ops.test('NormDispIncr',1e-8,10,0)

        time_h.append(ops.getTime())
        dg.append(ops.nodeDisp(node_id[0][0],  1))
        df.append(ops.nodeDisp(node_id[1][0],  1))
        dr.append(ops.nodeDisp(node_id[-1][0], 1))

    try: os.remove(tmp.name)
    except Exception: pass

    dg=np.array(dg); df=np.array(df); dr=np.array(dr); th=np.array(time_h)
    h = p['storey_height']

    conv_log.append({'stage':'time_history', 'total_steps':n_steps,
                     'n_fail':n_fail, 'n_newton':n_newton,
                     'n_krylov':n_krylov, 'n_modified':n_modified,
                     'convergence_rate_pct': round((1-n_fail/n_steps)*100,2)})

    return dict(time_h=th, dg=dg, df=df, dr=dr,
                drift_s1=(df-dg)/h, drift_s2=(dr-df)/h,
                conv_log=conv_log)

print("✓ Analysis engine loaded.")
print("  Functions: build_model | static_analysis | eigenvalue_analysis")
print("             run_pushover | run_time_history")

## Cell 5 — Post-processing, EDP computation, fragility curves, and plots
Computes all engineering demand parameters, generates per-building 8-panel figure, and fragility curves.

In [ ]:
# Cell 5: Post-processing, fragility analysis, and plotting functions

# ── EDP computation ───────────────────────────────────────────────────────────

def compute_edps(th, push, p, M_floor, W_total, V_static, T1):
    """Compute all Engineering Demand Parameters from analysis results."""
    PIDR1 = float(np.max(np.abs(th['drift_s1'])))
    PIDR2 = float(np.max(np.abs(th['drift_s2'])))
    PIDR  = max(PIDR1, PIDR2)

    omega1     = 2*np.pi/T1
    PFA_ground = p['Z'] * G
    PFA_f1     = omega1**2 * float(np.max(np.abs(th['df'])))
    PFA_roof   = omega1**2 * float(np.max(np.abs(th['dr'])))
    V_dyn      = M_floor*PFA_f1 + M_floor*PFA_roof

    # Spectral displacement (fundamental mode)
    Sd         = float(np.max(np.abs(th['dr'])))  # roof ~ mode 1

    # Storey-level peak accelerations
    # Numerical differentiation: a = d²u/dt²
    if len(th['time_h']) > 2:
        dt_arr = np.diff(th['time_h'])
        dt_avg = float(np.mean(dt_arr[dt_arr>0]))
        if dt_avg > 0:
            acc_f1   = np.gradient(np.gradient(th['df'],   dt_avg), dt_avg)
            acc_roof = np.gradient(np.gradient(th['dr'],   dt_avg), dt_avg)
            PFA_f1_num   = float(np.max(np.abs(acc_f1)))
            PFA_roof_num = float(np.max(np.abs(acc_roof)))
        else:
            PFA_f1_num = PFA_f1; PFA_roof_num = PFA_roof
    else:
        PFA_f1_num = PFA_f1; PFA_roof_num = PFA_roof

    damage_state = classify_damage(PIDR)
    frag_probs   = fragility_probability(p['era'], PIDR)

    return dict(
        PIDR1        = PIDR1,
        PIDR2        = PIDR2,
        PIDR_max     = PIDR,
        PFA_ground   = PFA_ground,
        PFA_f1       = PFA_f1,
        PFA_roof     = PFA_roof,
        PFA_f1_num   = PFA_f1_num,    # numerically differentiated
        PFA_roof_num = PFA_roof_num,
        amp_f1       = PFA_f1/PFA_ground,
        amp_roof     = PFA_roof/PFA_ground,
        Sd_m         = Sd,
        V_static     = V_static,
        V_dynamic    = V_dyn,
        W_total      = W_total,
        max_roof_mm  = Sd*1000,
        damage_state = damage_state,
        frag_probs   = frag_probs,
        drift_pass   = PIDR <= DRIFT_LIMIT,
        compliant    = PIDR <= DRIFT_LIMIT,
    )


# ── Full per-building assessment ──────────────────────────────────────────────

def assess_building(params, auto_approve=False):
    """Orchestrate all analysis stages for one building."""
    p = params
    print()
    print("=" * 65)
    print("  ASSESSING: {}".format(p['building_name']))
    print("=" * 65)

    p, approved = verification_checkpoint(p, auto_approve=auto_approve)
    if not approved:
        return None

    t0 = time.time()
    results = {'params': p}
    all_conv = []

    # Stage 1 -- Build model + gravity + mass
    try:
        node_id, M_floor, W_total, cl = build_model(p)
        all_conv.extend(cl)
        n_col = (p['num_bays']+1)*p['num_storeys']
        n_bm  = p['num_bays']*p['num_storeys']
        print("  Model built: {} nodes, {} elements".format(
              (p['num_bays']+1)*(p['num_storeys']+1), n_col+n_bm))
        grav_ok = cl[-1]['converged'] if cl else False
        print("  Gravity: {}".format("CONVERGED" if grav_ok else "WARNING"))
    except Exception as e:
        print("  Model build FAILED: {}".format(e))
        traceback.print_exc()
        return None

    # Stage 2 -- Static analysis
    sa = static_analysis(p, W_total)
    print("  Static: W={:.1f}kN  T1_code={:.3f}s  V={:.1f}kN  V/W={:.4f}".format(
          W_total, sa['T1_code'], sa['V_static'], sa['V_over_W']))

    # Stage 3 -- Eigenvalue
    ev = eigenvalue_analysis(p)
    if ev['success']:
        print("  Eigenvalue: T1={:.3f}s  T2={:.3f}s  (ratio={:.2f}x code)".format(
              ev['T1'], ev['T2'], ev['T1']/sa['T1_code']))
    else:
        print("  Eigenvalue FAILED: {}".format(ev['error']))
        return None

    # Stage 4 -- Pushover
    print("  Pushover: running...")
    try:
        push = run_pushover(node_id, p, W_total, sa['V_static'])
        all_conv.extend(push['conv_log'])
        duct = "{:.2f}".format(push['ductility']) if push['ductility'] else "N/A"
        print("  Pushover: V_max={:.1f}kN  ductility={}  steps_ok={}/200".format(
              push['V_max'], duct, len(push['roof_disp'])-1))
    except Exception as e:
        print("  Pushover failed: {} -- using null curve".format(e))
        push = dict(roof_disp=np.zeros(2), base_shear=np.zeros(2),
                    roof_drift=np.zeros(2), V_max=sa['V_static'],
                    d_yield_m=None, ductility=None, T_eff_s=None,
                    n_fail=0, conv_log=[])

    # Rebuild model (pushover modifies state)
    node_id, M_floor, W_total, _ = build_model(p)

    # Stage 5 -- Time-history
    print("  Time-history: running 4000 sub-steps...")
    try:
        th = run_time_history(node_id, p, ev['omega1'], ev['omega2'])
        all_conv.extend(th['conv_log'])
        cl_th = th['conv_log'][0]
        print("  TH done: n_fail={}/{}, Newton={}, Krylov={}, Modified={}".format(
              cl_th['n_fail'], cl_th['total_steps'],
              cl_th['n_newton'], cl_th['n_krylov'], cl_th['n_modified']))
        print("  Convergence rate: {:.1f}%".format(cl_th['convergence_rate_pct']))
    except Exception as e:
        print("  Time-history FAILED: {}".format(e))
        traceback.print_exc()
        return None

    # Stage 6 -- EDPs
    edp = compute_edps(th, push, p, M_floor, W_total, sa['V_static'], ev['T1'])

    elapsed = time.time() - t0
    print()
    print("  PIDR max   : {:.3f}%  (limit 1.5%) -> {}".format(
          edp['PIDR_max']*100, "PASS" if edp['drift_pass'] else "FAIL"))
    print("  Damage     : {}".format(edp['damage_state']))
    print("  Fragility  : Slight={:.0%}  Moderate={:.0%}  Extensive={:.0%}".format(
          edp['frag_probs'].get('Slight',0),
          edp['frag_probs'].get('Moderate',0),
          edp['frag_probs'].get('Extensive',0)))
    print("  PFA roof   : {:.3f}m/s2 ({:.3f}g)".format(
          edp['PFA_roof'], edp['PFA_roof']/G))
    print("  V_dynamic  : {:.1f}kN ({:.2f}x static)".format(
          edp['V_dynamic'], edp['V_dynamic']/sa['V_static']))
    print("  Elapsed    : {:.1f}s".format(elapsed))
    print("  RESULT     : {}".format(
          "COMPLIANT" if edp['compliant'] else "NON-COMPLIANT"))

    return dict(
        params=p, T1=ev['T1'], T2=ev['T2'],
        T1_code=sa['T1_code'], V_static=sa['V_static'],
        W_total=W_total, M_floor=M_floor,
        time_h=th['time_h'], df=th['df'], dr=th['dr'],
        drift_s1=th['drift_s1'], drift_s2=th['drift_s2'],
        pushover=push, convergence_log=all_conv,
        elapsed=elapsed, **edp,
    )


# ── Per-building 8-panel plot ─────────────────────────────────────────────────

def plot_single(r, filename=None):
    p   = r['params']
    th  = r['time_h']
    lim = DRIFT_LIMIT * 100
    col = ERA_COLOURS.get(p['era'], 'steelblue')

    fig = plt.figure(figsize=(17, 13))
    fig.suptitle(p['building_name'], fontsize=13, fontweight='bold', y=0.995)
    gs  = gridspec.GridSpec(3, 3, figure=fig, hspace=0.45, wspace=0.38)

    # 1 -- Roof displacement time history
    ax = fig.add_subplot(gs[0, 0])
    ax.plot(th, r['dr']*1000, color='steelblue', lw=1)
    ax.fill_between(th, r['dr']*1000, 0, alpha=0.15, color='steelblue')
    ax.axhline(0, color='k', lw=0.4)
    ax.set_xlabel('Time (s)');  ax.set_ylabel('Disp. (mm)')
    ax.set_title('Roof Displacement');  ax.grid(True, alpha=0.3)
    ax.text(0.98, 0.95, 'max={:.1f}mm'.format(r['max_roof_mm']),
            transform=ax.transAxes, ha='right', va='top', fontsize=8,
            bbox=dict(fc='white', alpha=0.7, edgecolor='none'))

    # 2 -- Drift time histories
    ax = fig.add_subplot(gs[0, 1])
    ax.plot(th, r['drift_s1']*100, color='tomato',     lw=1, label='Storey 1')
    ax.plot(th, r['drift_s2']*100, color='darkorange', lw=1, label='Storey 2')
    ax.axhline( lim, color='red', ls='--', lw=1.5, label='{:.0f}% limit'.format(lim))
    ax.axhline(-lim, color='red', ls='--', lw=1.5)
    ax.set_xlabel('Time (s)');  ax.set_ylabel('Drift (%)')
    ax.set_title('Inter-Storey Drift');  ax.legend(fontsize=8);  ax.grid(True, alpha=0.3)

    # 3 -- Pushover capacity curve
    ax = fig.add_subplot(gs[0, 2])
    push = r['pushover']
    ax.plot(push['roof_drift'], push['base_shear'], color=col, lw=2.5, label='Capacity')
    ax.axhline(r['V_static'], color='navy', ls='--', lw=1.5,
               label='V_stat={:.0f}kN'.format(r['V_static']))
    ax.axhline(push['V_max'], color='gray', ls=':', lw=1.2,
               label='V_max={:.0f}kN'.format(push['V_max']))
    if push['d_yield_m']:
        Hn = p['num_storeys'] * p['storey_height']
        ax.axvline(push['d_yield_m']/Hn*100, color='green', ls='-.', lw=1,
                   label='yield drift')
    ax.set_xlabel('Roof Drift (%)');  ax.set_ylabel('Base Shear (kN)')
    ax.set_title('Pushover Capacity Curve');  ax.legend(fontsize=7);  ax.grid(True, alpha=0.3)

    # 4 -- Peak drift storey profile
    ax = fig.add_subplot(gs[1, 0])
    dvals = [r['PIDR1']*100, r['PIDR2']*100]
    bcolors = ['#d9534f' if v>DRIFT_LIMIT else '#5cb85c' for v in (r['PIDR1'],r['PIDR2'])]
    bars = ax.barh([1,2], dvals, color=bcolors, edgecolor='white', height=0.5)
    ax.axvline(lim, color='red', ls='--', lw=1.5, label='1.5% limit')
    ax.set_xlabel('Peak Drift (%)');  ax.set_ylabel('Storey')
    ax.set_title('Peak Drift Profile');  ax.legend(fontsize=8);  ax.grid(True, alpha=0.3, axis='x')
    for bar,val in zip(bars,dvals):
        ax.text(bar.get_width()+0.01, bar.get_y()+bar.get_height()/2,
                '{:.3f}%'.format(val), va='center', fontsize=9)

    # 5 -- Floor acceleration profile
    ax = fig.add_subplot(gs[1, 1])
    floors = [0, 1, 2]
    pfas   = [r['PFA_ground']/G, r['PFA_f1']/G, r['PFA_roof']/G]
    pfas_n = [r['PFA_ground']/G, r['PFA_f1_num']/G, r['PFA_roof_num']/G]
    ax.barh(floors, pfas,   color='steelblue', alpha=0.7, height=0.35, label='omega2.u')
    ax.barh([f+0.35 for f in floors], pfas_n, color='navy', alpha=0.7, height=0.35, label='d2u/dt2')
    ax.set_xlabel('PFA (g)');  ax.set_title('Peak Floor Accelerations')
    ax.set_yticks([0.18,1.18,2.18]);  ax.set_yticklabels(['Ground','Floor 1','Roof'])
    ax.legend(fontsize=7);  ax.grid(True, alpha=0.3, axis='x')

    # 6 -- Damage state indicator
    ax = fig.add_subplot(gs[1, 2])
    ax.axis('off')
    ds_col = STATE_COLOURS.get(r['damage_state'], 'gray')
    circle = plt.Circle((0.5, 0.55), 0.36, color=ds_col, alpha=0.85)
    ax.add_patch(circle)
    ax.text(0.5, 0.57, r['damage_state'], ha='center', va='center',
            fontsize=14, fontweight='bold', color='white')
    ax.text(0.5, 0.22, 'HAZUS Damage State', ha='center', fontsize=9, color='#555')
    ax.text(0.5, 0.13, 'PIDR = {:.3f}%'.format(r['PIDR_max']*100),
            ha='center', fontsize=9, color='#555')
    # Fragility probabilities as mini bar chart
    fprobs = r['frag_probs']
    ds_names = list(fprobs.keys())
    ds_vals  = [fprobs[k] for k in ds_names]
    for i,(name,val) in enumerate(zip(ds_names,ds_vals)):
        y = 0.06 - i*0.00   # space is tight, annotate differently
    ax.set_xlim(0,1);  ax.set_ylim(0,1)

    # 7 -- Storey 1 hysteresis loop
    ax = fig.add_subplot(gs[2, 0])
    ax.plot(r['drift_s1']*100, r['df']*1000, color=col, lw=0.8, alpha=0.85)
    ax.axvline(0, color='k', lw=0.4);  ax.axhline(0, color='k', lw=0.4)
    ax.set_xlabel('Storey 1 Drift (%)');  ax.set_ylabel('Floor 1 Disp. (mm)')
    ax.set_title('Force-Displacement (Storey 1)');  ax.grid(True, alpha=0.3)

    # 8 -- Fragility probabilities bar chart
    ax = fig.add_subplot(gs[2, 1])
    fprobs   = r['frag_probs']
    ds_names = list(fprobs.keys())
    ds_vals  = [fprobs[k]*100 for k in ds_names]
    ds_cols  = [STATE_COLOURS.get(n,'gray') for n in ds_names]
    ax.bar(ds_names, ds_vals, color=ds_cols, edgecolor='white')
    ax.set_ylabel('P(DS >= ds | PIDR) %');  ax.set_title('Fragility Probabilities')
    ax.set_ylim(0, 105);  ax.grid(True, alpha=0.3, axis='y')
    for i,(name,val) in enumerate(zip(ds_names, ds_vals)):
        ax.text(i, val+2, '{:.0f}%'.format(val), ha='center', fontsize=9)

    # 9 -- Summary text
    ax = fig.add_subplot(gs[2, 2])
    ax.axis('off')
    push = r['pushover']
    duct_str = '{:.2f}'.format(push['ductility']) if push['ductility'] else 'N/A'
    comply   = 'COMPLIANT' if r['compliant'] else 'NON-COMPLIANT'
    lines = [
        '=' * 44,
        '  {} ERA'.format(p['era'].upper()),
        '=' * 44,
        "  f'c={} MPa  fy={} MPa".format(p['fc'], p['fy']),
        "  Col {}x{}mm  rho={:.1f}%  mu={}".format(
            int(p['col_b']*1000), int(p['col_h']*1000),
            p['col_rho']*100, p['mu']),
        '  Z={}  Site={}  Sp={}'.format(p['Z'],p['site_class'],p['Sp']),
        '-' * 44,
        '  T1 FEM = {:.3f}s  (code {:.3f}s)'.format(r['T1'], r['T1_code']),
        '  PIDR S1={:.3f}%  S2={:.3f}%'.format(r['PIDR1']*100,r['PIDR2']*100),
        '  Max PIDR = {:.3f}%  (limit 1.5%)'.format(r['PIDR_max']*100),
        '  Damage state: {}'.format(r['damage_state']),
        '-' * 44,
        '  V_max = {:.1f}kN  Ductility={}'.format(push['V_max'],duct_str),
        '  V_stat={:.1f}kN  V_dyn={:.1f}kN'.format(r['V_static'],r['V_dynamic']),
        '  Roof disp = {:.1f}mm'.format(r['max_roof_mm']),
        '  PFA amp (roof) = {:.2f}x'.format(r['amp_roof']),
        '=' * 44,
        '  AS1170.4: {}'.format(comply),
    ]
    ax.text(0.02, 0.98, '\n'.join(lines), transform=ax.transAxes,
            fontsize=8.5, va='top', fontfamily='monospace',
            bbox=dict(boxstyle='round', fc='lightyellow', ec='gray', alpha=0.9))

    fn = filename or 'results_{}.png'.format(p['era'].replace('-','').replace(' ','_'))
    plt.savefig(fn, dpi=150, bbox_inches='tight')
    plt.show();  plt.close()
    print("  Plot saved: {}".format(fn))
    return fn


# ── Fragility curves plot ─────────────────────────────────────────────────────

def plot_fragility_curves(results_list):
    """
    Lognormal fragility curves showing P(DS >= ds | PIDR) for each building.
    One subplot per damage state, all buildings overlaid.
    Vertical line at each building's actual PIDR.
    """
    pidr_range = np.logspace(-3, -0.5, 300)   # 0.001 to 0.316 (0.1% to 31.6%)

    ds_names = ['Slight','Moderate','Extensive','Complete']
    fig, axes = plt.subplots(2, 2, figsize=(13, 10))
    fig.suptitle('Seismic Fragility Curves\nP(DS >= ds | PIDR) | Newcastle RC Residential Buildings',
                 fontsize=12, fontweight='bold')
    axes = axes.flatten()

    for ax, ds_name in zip(axes, ds_names):
        ax.set_xlabel('Peak Inter-Storey Drift Ratio (%)')
        ax.set_ylabel('Probability of Exceedance')
        ax.set_title('{} Damage State'.format(ds_name))
        ax.set_xlim(0, 5.0);  ax.set_ylim(0, 1.05)
        ax.axvline(DRIFT_LIMIT*100, color='red', ls=':', lw=1.2,
                   label='AS1170.4 limit 1.5%')
        ax.grid(True, alpha=0.3)

        for r in results_list:
            era = r['params']['era']
            col = ERA_COLOURS.get(era, 'gray')
            frag_params = FRAGILITY.get(era, {})
            if ds_name in frag_params:
                median, beta = frag_params[ds_name]
                prob = stats.norm.cdf(np.log(pidr_range/median)/beta)
                ax.plot(pidr_range*100, prob, color=col, lw=2.5,
                        label=era, alpha=0.9)
                # Mark actual PIDR
                actual = r['PIDR_max']
                if actual > 0 and median > 0:
                    p_actual = float(stats.norm.cdf(np.log(actual/median)/beta))
                    ax.scatter([actual*100], [p_actual],
                               color=col, zorder=5, s=80,
                               marker='D', edgecolors='white', linewidths=0.8)
                    ax.annotate('{:.2f}%'.format(actual*100),
                                (actual*100, p_actual),
                                xytext=(5, 0), textcoords='offset points',
                                fontsize=8, color=col)

        handles = [Line2D([0],[0],color=ERA_COLOURS[e],lw=2.5,label=e)
                   for e in ['pre-1990','post-1990','post-2010']]
        handles.append(Line2D([0],[0],color='red',ls=':',lw=1.2,label='1.5% limit'))
        ax.legend(handles=handles, fontsize=8, loc='upper left')

    plt.tight_layout()
    fn = 'fragility_curves.png'
    plt.savefig(fn, dpi=150, bbox_inches='tight')
    plt.show();  plt.close()
    print("  Fragility curves saved: {}".format(fn))
    return fn


# ── Multi-building comparison plot ────────────────────────────────────────────

def plot_comparison(results_list):
    """7-panel comparison chart across all assessed buildings."""
    if len(results_list) < 2:
        print("  Need >= 2 buildings for comparison.")
        return

    n      = len(results_list)
    names  = ['B{}'.format(i+1) for i in range(n)]
    eras   = [r['params']['era'] for r in results_list]
    colors = [ERA_COLOURS.get(e,'gray') for e in eras]

    fig    = plt.figure(figsize=(18, 12))
    fig.suptitle('Seismic Vulnerability Comparison -- Newcastle RC Residential Buildings\n'
                 'AS1170.4:2007  |  UTS Engineering Graduate Project PG (42003)',
                 fontsize=12, fontweight='bold')
    gs     = gridspec.GridSpec(3, 3, figure=fig, hspace=0.48, wspace=0.38)

    # 1 -- PIDR bar chart with damage state colour coding
    ax = fig.add_subplot(gs[0, 0])
    bar_cols = [STATE_COLOURS.get(r['damage_state'],'gray') for r in results_list]
    bars = ax.bar(names, [r['PIDR_max']*100 for r in results_list],
                  color=bar_cols, edgecolor='white', linewidth=1.5)
    ax.axhline(DRIFT_LIMIT*100, color='red', ls='--', lw=2, label='1.5% limit')
    ax.set_ylabel('Peak PIDR (%)');  ax.set_title('Peak Inter-Storey Drift')
    ax.legend(fontsize=8);  ax.grid(True, alpha=0.3, axis='y')
    for bar,r in zip(bars,results_list):
        ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.02,
                '{:.3f}%\n{}'.format(r['PIDR_max']*100, r['damage_state']),
                ha='center', fontsize=8)

    # 2 -- Period comparison: FEM vs code
    ax = fig.add_subplot(gs[0, 1])
    xpos = np.arange(n);  w = 0.35
    b1 = ax.bar(xpos-w/2, [r['T1'] for r in results_list], w,
                color=colors, label='T1 FEM', alpha=0.9)
    b2 = ax.bar(xpos+w/2, [r['T1_code'] for r in results_list], w,
                color='lightgray', edgecolor='gray', label='T1 code')
    ax.set_xticks(xpos);  ax.set_xticklabels(names)
    ax.set_ylabel('Period (s)');  ax.set_title('Fundamental Period T1')
    ax.legend(fontsize=8);  ax.grid(True, alpha=0.3, axis='y')
    for bar,r in zip(b1, results_list):
        ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.005,
                '{:.3f}s'.format(r['T1']), ha='center', fontsize=8)

    # 3 -- Static vs dynamic base shear
    ax = fig.add_subplot(gs[0, 2])
    b1 = ax.bar(xpos-w/2, [r['V_static']  for r in results_list], w,
                color=colors, alpha=1.0, label='V static')
    b2 = ax.bar(xpos+w/2, [r['V_dynamic'] for r in results_list], w,
                color=colors, alpha=0.35, label='V dynamic')
    ax.set_xticks(xpos);  ax.set_xticklabels(names)
    ax.set_ylabel('Base Shear (kN)');  ax.set_title('Base Shear Comparison')
    ax.legend(fontsize=8);  ax.grid(True, alpha=0.3, axis='y')

    # 4 -- Overlaid pushover curves
    ax = fig.add_subplot(gs[1, 0])
    for r,col in zip(results_list,colors):
        push = r['pushover']
        ax.plot(push['roof_drift'], push['base_shear'],
                color=col, lw=2.5, label=r['params']['era'])
        ax.scatter([push['roof_drift'][-1]], [push['base_shear'][-1]],
                   color=col, s=60, zorder=5)
    ax.axvline(DRIFT_LIMIT*100/r['params']['num_storeys'], color='red',
               ls=':', lw=1, label='storey limit')
    ax.set_xlabel('Roof Drift (%)');  ax.set_ylabel('Base Shear (kN)')
    ax.set_title('Pushover Capacity Curves');  ax.legend(fontsize=8);  ax.grid(True, alpha=0.3)

    # 5 -- Fragility exceedance at actual PIDR
    ax = fig.add_subplot(gs[1, 1])
    ds_names = ['Slight','Moderate','Extensive','Complete']
    xpos2    = np.arange(len(ds_names));  w2 = 0.8/n
    for i,(r,col) in enumerate(zip(results_list,colors)):
        probs = [r['frag_probs'].get(ds,0)*100 for ds in ds_names]
        ax.bar(xpos2+i*w2 - 0.4+w2/2, probs, w2,
               color=col, alpha=0.85, label=r['params']['era'])
    ax.set_xticks(xpos2);  ax.set_xticklabels(ds_names, fontsize=9)
    ax.set_ylabel('P(DS >= ds | actual PIDR) %')
    ax.set_title('Fragility Exceedance at Actual PIDR')
    ax.legend(fontsize=7);  ax.grid(True, alpha=0.3, axis='y');  ax.set_ylim(0,105)

    # 6 -- Floor acceleration amplification
    ax = fig.add_subplot(gs[1, 2])
    floor_labels = ['Ground','Floor 1','Roof'];  w3 = 0.8/n
    for i,(r,col) in enumerate(zip(results_list,colors)):
        pfas = [r['PFA_ground']/G, r['PFA_f1']/G, r['PFA_roof']/G]
        ax.bar(xpos2+i*w3 - 0.4+w3/2, pfas, w3, color=col, alpha=0.85,
               label=r['params']['era'])
    ax.set_xticks(xpos2);  ax.set_xticklabels(floor_labels)
    ax.set_ylabel('PFA (g)');  ax.set_title('Peak Floor Accelerations')
    ax.legend(fontsize=7);  ax.grid(True, alpha=0.3, axis='y')

    # 7 -- Summary table spanning bottom row
    ax = fig.add_subplot(gs[2, :])
    ax.axis('off')
    col_labels = ['Building','Era',"f'c (MPa)",'fy (MPa)','mu',
                  'T1 FEM (s)','T1 code (s)','Max PIDR (%)','V_static (kN)',
                  'V_max push. (kN)','Ductility','Damage State','AS1170.4']
    table_data = []
    for r in results_list:
        p    = r['params']
        push = r['pushover']
        duct = '{:.2f}'.format(push['ductility']) if push['ductility'] else 'N/A'
        table_data.append([
            p['building_name'].split('--')[0].strip(),
            p['era'],
            str(p['fc']), str(p['fy']), str(p['mu']),
            '{:.3f}'.format(r['T1']),
            '{:.3f}'.format(r['T1_code']),
            '{:.3f}'.format(r['PIDR_max']*100),
            '{:.1f}'.format(r['V_static']),
            '{:.1f}'.format(push['V_max']),
            duct,
            r['damage_state'],
            'PASS' if r['compliant'] else 'FAIL',
        ])
    tbl = ax.table(cellText=table_data, colLabels=col_labels,
                   cellLoc='center', loc='center', bbox=[0,0,1,1])
    tbl.auto_set_font_size(False);  tbl.set_fontsize(8.5)
    tbl.auto_set_column_width(col=list(range(len(col_labels))))
    for i,r in enumerate(results_list):
        bg = '#d4edda' if r['compliant'] else '#f8d7da'
        tbl[i+1, 12].set_facecolor(bg)
        ds_bg = STATE_COLOURS.get(r['damage_state'],'white') + '55'
        tbl[i+1, 11].set_facecolor(ds_bg)
        era_bg = ERA_COLOURS.get(r['params']['era'],'#eee') + '33'
        tbl[i+1, 1].set_facecolor(era_bg)
    for j in range(len(col_labels)):
        tbl[0,j].set_facecolor('#34495e');  tbl[0,j].set_text_props(color='white')

    patches = [Patch(color=ERA_COLOURS[e], label=e) for e in ERA_DEFAULTS]
    fig.legend(handles=patches, loc='lower center', ncol=3,
               fontsize=9, bbox_to_anchor=(0.5,0.002))

    fn = 'comparison_all_buildings.png'
    plt.savefig(fn, dpi=150, bbox_inches='tight')
    plt.show();  plt.close()
    print("  Comparison chart saved: {}".format(fn))
    return fn


# ── Convergence report plot ───────────────────────────────────────────────────

def plot_convergence_report(results_list):
    """Bar chart showing convergence statistics across all buildings and stages."""
    fig, axes = plt.subplots(1, 2, figsize=(13, 5))
    fig.suptitle('Convergence Report -- Analysis Quality Metrics', fontsize=12, fontweight='bold')

    names  = ['B{} ({})'.format(i+1, r['params']['era']) for i,r in enumerate(results_list)]
    colors = [ERA_COLOURS.get(r['params']['era'],'gray') for r in results_list]

    # Convergence rates
    ax = axes[0]
    rates = []
    for r in results_list:
        for log in r.get('convergence_log',[]):
            if log.get('stage') == 'time_history':
                rates.append(log.get('convergence_rate_pct', 100))
                break
        else:
            rates.append(100.0)
    bars = ax.bar(names, rates, color=colors, edgecolor='white')
    ax.axhline(99, color='green', ls='--', lw=1.5, label='99% threshold')
    ax.set_ylim(90, 101)
    ax.set_ylabel('Convergence Rate (%)');  ax.set_title('Time-History Convergence Rate')
    ax.legend(fontsize=8);  ax.grid(True, alpha=0.3, axis='y')
    for bar,val in zip(bars,rates):
        ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.05,
                '{:.2f}%'.format(val), ha='center', fontsize=9)

    # Fallback steps
    ax = axes[1]
    xpos = np.arange(len(results_list));  w = 0.25
    for i, alg in enumerate(['n_newton','n_krylov','n_modified']):
        vals = []
        for r in results_list:
            for log in r.get('convergence_log',[]):
                if log.get('stage') == 'time_history':
                    vals.append(log.get(alg, 0))
                    break
            else:
                vals.append(0)
        label = ['Newton','KrylovNewton','ModifiedNewton'][i]
        col   = ['steelblue','orange','tomato'][i]
        ax.bar(xpos + i*w, vals, w, label=label, color=col, alpha=0.85)
    ax.set_xticks(xpos+w);  ax.set_xticklabels(names, fontsize=8)
    ax.set_ylabel('Steps');  ax.set_title('Algorithm Usage -- Time-History')
    ax.legend(fontsize=8);  ax.grid(True, alpha=0.3, axis='y')

    plt.tight_layout()
    fn = 'convergence_report.png'
    plt.savefig(fn, dpi=150, bbox_inches='tight')
    plt.show();  plt.close()
    print("  Convergence report saved: {}".format(fn))
    return fn

print("✓ Post-processing and plotting functions loaded.")
print("  Functions: compute_edps | assess_building | plot_single")
print("             plot_fragility_curves | plot_comparison | plot_convergence_report")

## Cell 6 — Run all three case study buildings
Runs Buildings 1, 2, and 3 in sequence. Auto-approves the preset verified parameters.  
Generates per-building plots, fragility curves, comparison chart, convergence report, and JSON report.  
**Expected total runtime: ~8–12 minutes on Google Colab.**

In [ ]:
# Cell 6: Run all three case study buildings
# ─────────────────────────────────────────────────────────────────────────────
# Expected runtime: ~8-12 minutes total on Google Colab
# All three buildings run automatically -- no prompts required
# ─────────────────────────────────────────────────────────────────────────────

print("╔" + "═"*63 + "╗")
print("║  SEISMIC VULNERABILITY ASSESSMENT -- ALL THREE BUILDINGS      ║")
print("║  UTS Engineering Graduate Project PG (42003)                 ║")
print("║  Kabish Jung Thapa | Supervisor: Prof. Jianchun Li           ║")
print("╚" + "═"*63 + "╝")
print()
print("  Running Buildings 1, 2, 3 in sequence...")
print("  Auto-approving verified case study parameters.")
print()

RESULTS = []

for i, building_params in enumerate(BUILDINGS, 1):
    result = assess_building(building_params, auto_approve=True)
    if result:
        RESULTS.append(result)
        # Per-building 8-panel plot
        fn = 'results_building{}_{}.png'.format(
              i, building_params['era'].replace('-','').replace(' ','_'))
        plot_single(result, filename=fn)
        print()

# ── Comparison outputs ────────────────────────────────────────────────────────
if len(RESULTS) >= 2:
    plot_comparison(RESULTS)
    plot_fragility_curves(RESULTS)
    plot_convergence_report(RESULTS)

# ── Summary table ─────────────────────────────────────────────────────────────
print()
print("=" * 90)
print("  FINAL SUMMARY TABLE -- AS1170.4:2007 COMPLIANCE")
print("=" * 90)
hdr = "  {:<38}  {:>8}  {:>7}  {:>8}  {:<12}  {:>10}  {:>10}"
print(hdr.format('Building','T1 FEM','PIDR','V_stat','Damage','Ductility','AS1170.4'))
print("  " + "-"*84)
for r in RESULTS:
    push = r['pushover']
    duct = "{:.2f}".format(push['ductility']) if push['ductility'] else "N/A"
    print(hdr.format(
        r['params']['building_name'][:38],
        "{:.3f}s".format(r['T1']),
        "{:.3f}%".format(r['PIDR_max']*100),
        "{:.1f}kN".format(r['V_static']),
        r['damage_state'],
        duct,
        "PASS" if r['compliant'] else "** FAIL **",
    ))
print("  " + "-"*84)
print(hdr.format('AS1170.4 Limit','--','1.500%','--','--','--','--'))
print("=" * 90)

# ── JSON report ───────────────────────────────────────────────────────────────
def save_comprehensive_report(results_list):
    report = {
        "project":     "UTS Engineering Graduate Project PG (42003)",
        "student":     "Kabish Jung Thapa (25631413)",
        "supervisor":  "Prof. Jianchun Li",
        "generated":   datetime.now().strftime("%Y-%m-%d %H:%M"),
        "standard":    "AS1170.4:2007",
        "location":    "Newcastle, NSW  (Z=0.11, Site De)",
        "llm_mode":    "Demo mode -- keyword extraction, no API key required",
        "buildings":   [],
    }

    for r in results_list:
        p    = r['params']
        push = r['pushover']

        # Convergence summary from log
        conv_summary = {}
        for log in r.get('convergence_log',[]):
            stage = log.get('stage','')
            if stage == 'gravity':
                conv_summary['gravity_converged'] = log.get('converged', False)
            elif stage == 'pushover':
                conv_summary['pushover_steps']   = log.get('n_steps_complete',0)
                conv_summary['pushover_n_fail']  = log.get('n_fail',0)
            elif stage == 'time_history':
                conv_summary['th_total_steps']    = log.get('total_steps',0)
                conv_summary['th_n_fail']         = log.get('n_fail',0)
                conv_summary['th_conv_rate_pct']  = log.get('convergence_rate_pct',0)
                conv_summary['th_newton_steps']   = log.get('n_newton',0)
                conv_summary['th_krylov_steps']   = log.get('n_krylov',0)
                conv_summary['th_modified_steps'] = log.get('n_modified',0)

        report['buildings'].append({
            "building_name": p['building_name'],
            "era":           p['era'],
            "structural_parameters": {
                "fc_MPa":    p['fc'],
                "fy_MPa":    p['fy'],
                "col_mm":    "{}x{}".format(int(p['col_b']*1000),int(p['col_h']*1000)),
                "beam_mm":   "{}x{}".format(int(p['beam_b']*1000),int(p['beam_h']*1000)),
                "col_rho_%": round(p['col_rho']*100, 2),
                "mu":        p['mu'],   "Sp":  p['Sp'],
                "Z":         p['Z'],    "site":p['site_class'],
            },
            "loads": {
                "seismic_weight_kN": round(r['W_total'],1),
                "dead_kPa":          p['dead_load'],
                "live_kPa":          p['live_load'],
                "mass_per_floor_kNs2m": round(r['M_floor'],3),
            },
            "periods": {
                "T1_FEM_s":  round(r['T1'],4),
                "T2_FEM_s":  round(r['T2'],4),
                "T1_code_s": round(r['T1_code'],4),
                "T1_ratio":  round(r['T1']/r['T1_code'],3),
            },
            "static_analysis": {
                "V_static_kN": round(r['V_static'],2),
                "V_over_W":    round(r['V_static']/r['W_total'],4),
            },
            "pushover": {
                "V_max_kN":    round(push['V_max'],2),
                "ductility":   round(push['ductility'],3) if push['ductility'] else None,
                "T_eff_s":     round(push['T_eff_s'],4) if push['T_eff_s'] else None,
                "n_steps":     len(push['roof_disp'])-1,
            },
            "time_history_edps": {
                "PIDR_s1_%":        round(r['PIDR1']*100,4),
                "PIDR_s2_%":        round(r['PIDR2']*100,4),
                "PIDR_max_%":       round(r['PIDR_max']*100,4),
                "governing_storey": 1 if r['PIDR1']>=r['PIDR2'] else 2,
                "drift_limit_%":    1.5,
                "drift_pass":       r['drift_pass'],
                "PFA_ground_g":     round(r['PFA_ground']/G,4),
                "PFA_floor1_g":     round(r['PFA_f1']/G,4),
                "PFA_roof_g":       round(r['PFA_roof']/G,4),
                "PFA_floor1_num_g": round(r['PFA_f1_num']/G,4),
                "PFA_roof_num_g":   round(r['PFA_roof_num']/G,4),
                "amp_floor1_x":     round(r['amp_f1'],3),
                "amp_roof_x":       round(r['amp_roof'],3),
                "Sd_m":             round(r['Sd_m'],5),
                "max_roof_disp_mm": round(r['max_roof_mm'],3),
                "V_dynamic_kN":     round(r['V_dynamic'],2),
                "V_dyn_stat_ratio": round(r['V_dynamic']/r['V_static'],3),
            },
            "damage_assessment": {
                "hazus_damage_state": r['damage_state'],
                "as1170_4_result":    "COMPLIANT" if r['compliant'] else "NON-COMPLIANT",
                "fragility_probs_at_actual_PIDR": {
                    k: round(v,4) for k,v in r['frag_probs'].items()
                },
            },
            "convergence_summary": conv_summary,
            "run_time_s":  round(r['elapsed'],1),
            "assumptions": p.get('assumptions',[]),
        })

    # Cross-building comparison
    if len(results_list) > 1:
        max_pidr_r = max(results_list, key=lambda x: x['PIDR_max'])
        min_pidr_r = min(results_list, key=lambda x: x['PIDR_max'])
        max_vmax_r = max(results_list, key=lambda x: x['pushover']['V_max'])
        report["cross_building_comparison"] = {
            "most_vulnerable_era":   max_pidr_r['params']['era'],
            "most_vulnerable_PIDR_%":round(max_pidr_r['PIDR_max']*100,3),
            "least_vulnerable_era":  min_pidr_r['params']['era'],
            "least_vulnerable_PIDR_%":round(min_pidr_r['PIDR_max']*100,3),
            "highest_capacity_era":  max_vmax_r['params']['era'],
            "highest_V_max_kN":      round(max_vmax_r['pushover']['V_max'],2),
            "all_compliant":         all(r['compliant'] for r in results_list),
            "pidr_range_%":          [round(min(r['PIDR_max'] for r in results_list)*100,3),
                                      round(max(r['PIDR_max'] for r in results_list)*100,3)],
            "V_static_range_kN":     [round(min(r['V_static'] for r in results_list),1),
                                      round(max(r['V_static'] for r in results_list),1)],
            "key_finding": (
                "Pre-1990 buildings attract {:.1f}x the design base shear of post-2010 "
                "buildings due to lower ductility factor (mu=2.0 vs 4.0), yet have lower "
                "structural capacity (f'c=20 MPa, non-ductile detailing). This represents "
                "the highest vulnerability category in the Newcastle residential stock."
            ).format(
                results_list[0]['V_static'] / results_list[-1]['V_static']
                if results_list[-1]['V_static'] > 0 else 1.0
            ),
        }

    fn = 'seismic_assessment_report.json'
    with open(fn,'w') as f:
        json.dump(report, f, indent=2)
    print("  JSON report saved: {}".format(fn))
    return fn

save_comprehensive_report(RESULTS)

# ── Final banner ───────────────────────────────────────────────────────────────
print()
print("╔" + "═"*63 + "╗")
print("║  ALL ASSESSMENTS COMPLETE                                    ║")
for i,r in enumerate(RESULTS,1):
    name   = r['params']['building_name'][:40]
    status = 'PASS' if r['compliant'] else 'FAIL'
    print("║  B{} [{}]  {:<54}║".format(i,status,name))
print("╚" + "═"*63 + "╝")
print()
print("  Output files:")
for i,r in enumerate(RESULTS,1):
    era = r['params']['era'].replace('-','').replace(' ','_')
    print("  |-- results_building{}_{}.png".format(i,era))
print("  |-- comparison_all_buildings.png")
print("  |-- fragility_curves.png")
print("  |-- convergence_report.png")
print("  |-- seismic_assessment_report.json")
print()
print("  Download: Files panel in left Colab sidebar -> right-click each file")

## Cell 7 (Optional) — Assess your own building
Run this cell separately to assess a custom building by description.  
You will be prompted to approve the extracted parameters before analysis runs.

In [ ]:
# Cell 7 (Optional): Assess a custom building by description
# Run this cell independently after Cell 5 has been executed.

# ── Option A: Free text description ──────────────────────────────────────────
DESCRIPTION = """
A 2-storey reinforced concrete residential building in Newcastle, NSW,
built approximately 1975. Floor plan approximately 12 metres by 8 metres.
Non-ductile design, no seismic detailing -- typical pre-1990 construction.
"""

# ── Option B: Override specific parameters after extraction ───────────────────
OVERRIDES = {
    # Uncomment any line to override the extracted value:
    # 'fc':        25.0,    # MPa -- if you know the actual concrete strength
    # 'fy':       300.0,    # MPa -- if reinforcement is known
    # 'num_storeys': 3,     # override storey count
    # 'Z':          0.08,   # override seismic zone
}

# ── Run ───────────────────────────────────────────────────────────────────────
print("Extracting parameters from description...")
custom_params = extract_parameters(DESCRIPTION)

# Apply any manual overrides
for key, val in OVERRIDES.items():
    if key in custom_params:
        custom_params[key] = val
        custom_params.setdefault('assumptions',[]).append(
            "User override: {}={}".format(key,val))
        print("  Override applied: {}={}".format(key,val))

# Run with manual verification (auto_approve=False prompts you to Y/E/N)
custom_result = assess_building(custom_params, auto_approve=False)

if custom_result:
    plot_single(custom_result, filename='results_custom.png')
    # Add to RESULTS for combined comparison (optional)
    # RESULTS.append(custom_result)
    print("Custom building assessment complete.")
else:
    print("Assessment was rejected or failed.")